Copyright 2026 Snowflake Inc.  
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# From Messy Data to Instant Answers with Snowflake CoCo and CoWork

Northwind Outfitters sells shoes, clothing, accessories, and home goods online in the US, Canada, and Europe. Leadership wants to know:

> **Which products are customers returning most, and why?**

The data lives in three places that don't agree: an order export from the store, return events from three different systems (the returns portal, the warehouse, and customer support), and free-text comments customers left when they started a return. Some of those comments are in German, French, and Spanish.

In this lab you will:

1. Ask an AI agent the question over the raw data and get a confident, **wrong** answer.
2. Use **CoCo** to find out why, then fix what "a return" means and clean up the mess.
3. Use AI functions to translate customer comments and tag a reason on each one.
4. Give an agent both the numbers (Cortex Analyst) and the customers' own words (Cortex Search).
5. Ask the same question in **CoWork** and get an answer you can trust, with quotes.

The data is 2,000 order lines: big enough that you can't eyeball the answer, small enough that every step runs in seconds. The same steps work unchanged on millions of rows.

You'll see a lot of SQL. You don't need to understand every line: CoCo writes it, and each step ends with a check that tells you whether the numbers are right.

## STEP 1: Set up the workspace

You created this workspace from the lab's GitHub repo in the guide. You're in the notebook now, and you'll stay here through STEP 8. After that you'll finish the lab in Snowsight.

1. Click **Connect** next to the Run button and select **Create and connect**.
2. Wait for the status bar at the bottom of the notebook to show **Connected**.

Run each cell when the instructions reach it, with **Run** on the cell or **Shift+Enter**. Every object in this lab uses a fully qualified name, so you don't need to pick a database or schema.

Now run the **`setup`** cell. It creates the lab database `ONLINE_RETURNS` with three schemas and a stage:

| Schema | Holds |
|---|---|
| `RAW` | Files exactly as received |
| `CURATED` | Cleaned tables, one row per order line |
| `ANALYTICS` | The semantic views, search service, and agents |

In [ ]:
-- SETUP: create the lab database, schemas, and stage.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE DATABASE ONLINE_RETURNS;
CREATE OR REPLACE SCHEMA ONLINE_RETURNS.RAW;
CREATE OR REPLACE SCHEMA ONLINE_RETURNS.CURATED;
CREATE OR REPLACE SCHEMA ONLINE_RETURNS.ANALYTICS;

CREATE OR REPLACE STAGE ONLINE_RETURNS.RAW.LAB_FILES;

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

## STEP 2: Load the raw files

Before you can fix the data, you need it in Snowflake exactly as the source systems sent it. Keeping an untouched copy means you can always check a cleaned number against the original.

The repo's `data/` folder has three files:

| File | What it is |
|---|---|
| `orders.csv` | The store's order export: 2,000 order lines, one row per item bought. Each row has the category, region, price, order date, and the store system's `order_status`. |
| `return_events.json` | The log of each return as it moves along: requested in the returns portal, received or rejected by the warehouse, and refunded or exchanged by customer support. Each step is one event with a timestamp and the system that logged it, 1,900 events in all. |
| `return_comments.json` | What customers typed in the returns portal about why they're sending the item back: 775 comments, about a third of them in German, French, or Spanish. |

Run the **`upload_files`** cell. It finds each file in your workspace and copies it, unchanged, to the stage `ONLINE_RETURNS.RAW.LAB_FILES`. You should see three `Uploaded` lines.

In [ ]:
from snowflake.snowpark.context import get_active_session
from pathlib import Path

session = get_active_session()

for filename in ["orders.csv", "return_events.json", "return_comments.json"]:
    matches = list(Path(".").glob(f"**/data/{filename}"))
    if not matches:
        raise FileNotFoundError(f"Could not find data/{filename}")
    session.file.put(
        str(matches[0]),
        "@ONLINE_RETURNS.RAW.LAB_FILES",
        auto_compress=False,
        overwrite=True,
    )
    print(f"Uploaded {filename}")

Now run the **`load_raw`** cell. It creates three raw tables and copies each file from the stage into one of them:

- `RAW.ORDERS` has one column per CSV field, and **every column is text** on purpose. Prices like `$1,049.00` and dates in four formats stay exactly as they arrived, so nothing gets lost or silently converted.
- `RAW.RETURN_EVENTS` and `RAW.RETURN_COMMENTS` each hold their whole JSON file in a single `VARIANT` column called `payload`. You'll unpack it with `FLATTEN` later.

The last query counts what landed. Expect **2,000** order lines, **1,900** return events, and **775** comments, matching the file descriptions above.

In [ ]:
-- Load the files into raw tables without changing anything.
CREATE OR REPLACE TABLE ONLINE_RETURNS.RAW.ORDERS (
  order_line_id VARCHAR,
  order_id VARCHAR,
  customer_id VARCHAR,
  category VARCHAR,
  region VARCHAR,
  item_price VARCHAR,
  order_date VARCHAR,
  order_status VARCHAR
);
CREATE OR REPLACE TABLE ONLINE_RETURNS.RAW.RETURN_EVENTS (payload VARIANT);
CREATE OR REPLACE TABLE ONLINE_RETURNS.RAW.RETURN_COMMENTS (payload VARIANT);

COPY INTO ONLINE_RETURNS.RAW.ORDERS
FROM @ONLINE_RETURNS.RAW.LAB_FILES
FILES = ('orders.csv')
FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');

COPY INTO ONLINE_RETURNS.RAW.RETURN_EVENTS
FROM @ONLINE_RETURNS.RAW.LAB_FILES
FILES = ('return_events.json')
FILE_FORMAT = (TYPE = JSON);

COPY INTO ONLINE_RETURNS.RAW.RETURN_COMMENTS
FROM @ONLINE_RETURNS.RAW.LAB_FILES
FILES = ('return_comments.json')
FILE_FORMAT = (TYPE = JSON);

-- Expect 2,000 order lines, 1,900 return events, 775 comments.
SELECT
  (SELECT COUNT(*) FROM ONLINE_RETURNS.RAW.ORDERS) AS order_lines,
  (SELECT ARRAY_SIZE(payload) FROM ONLINE_RETURNS.RAW.RETURN_EVENTS) AS return_events,
  (SELECT ARRAY_SIZE(payload) FROM ONLINE_RETURNS.RAW.RETURN_COMMENTS) AS comments;

## STEP 3: See the mess

Before anyone asks an AI agent about this data, look at it yourself. Run the **`peek_mess`** cell. It shows four columns from the raw data: `category`, `region`, and `order_status` from the order export, and `return_status` from the return events. For each one, you'll see every distinct value stored and how many rows have it. The output has three columns:

| Column | What it shows |
|---|---|
| `column_name` | Which raw column the row describes. For return events, it also names the system that logged it (portal, warehouse, or support). |
| `value` | One distinct value exactly as stored, capitals and all |
| `rows` | How many rows have that value |

In [ ]:
SELECT 'category' AS column_name, category AS value, COUNT(*) AS rows FROM ONLINE_RETURNS.RAW.ORDERS GROUP BY 2
UNION ALL
SELECT 'region', region, COUNT(*) FROM ONLINE_RETURNS.RAW.ORDERS GROUP BY 2
UNION ALL
SELECT 'order_status', order_status, COUNT(*) FROM ONLINE_RETURNS.RAW.ORDERS GROUP BY 2
UNION ALL
SELECT 'event return_status (' || value:source::VARCHAR || ')', value:return_status::VARCHAR, COUNT(*)
FROM ONLINE_RETURNS.RAW.RETURN_EVENTS, LATERAL FLATTEN(input => payload) GROUP BY 1, 2
ORDER BY column_name, value;

What you're looking at:

| Column | Distinct values | What's going on |
|---|---:|---|
| `category` | 16 | Four real categories, each spelled several ways. Footwear is `footwear`, `FOOTWEAR`, `Footwear`, and `Shoes`. |
| `region` | 13 | Four real regions. EMEA is also `Europe` and `Europe/ME/Africa`. |
| `order_status` | 8 | `delivered`, `returned`, `shipped`, and `cancelled`, in mixed capitals. |
| event `return_status` | 9 | Three systems, three vocabularies: the warehouse says `refund_issued` where support says `refunded`. |

So there are **two** records of returns that may disagree: the export's `order_status`, and the trail of return events. Keep that in mind for the next step.

## STEP 4: Get a quick answer

The fastest thing you could do is point an AI agent straight at the order export and ask. Run the **`quick_agent`** cell. It creates two objects:

- `QUICK_RETURNS_SV`, a **semantic view** that tells AI what the export's columns mean. This one counts a line as returned whenever `order_status` says `returned`.
- `RETURNS_AGENT`, an **agent** that answers questions by writing SQL against that semantic view.

This is a first attempt on purpose. In STEP 10 you'll build a better semantic view, and in STEP 11 you'll edit this same agent to use it.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW ONLINE_RETURNS.ANALYTICS.QUICK_RETURNS_SV
  TABLES (
    orders AS ONLINE_RETURNS.RAW.ORDERS PRIMARY KEY (order_line_id)
      COMMENT = 'Order lines exactly as exported from the store system'
  )
  DIMENSIONS (
    orders.category AS category,
    orders.region AS region,
    orders.order_status AS order_status
  )
  METRICS (
    orders.order_lines AS COUNT(order_line_id),
    orders.returned_lines AS COUNT_IF(LOWER(order_status) = 'returned'),
    return_rate AS DIV0(orders.returned_lines, orders.order_lines)
  )
  COMMENT = 'Quick semantic view over the raw export';

CREATE OR REPLACE AGENT ONLINE_RETURNS.ANALYTICS.RETURNS_AGENT
  COMMENT = 'Returns agent, first version over raw data. Deliberately not trustworthy.'
  FROM SPECIFICATION
$$
models:
  orchestration: auto
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: quick_returns
      description: Order lines from the raw store export.
tool_resources:
  quick_returns:
    semantic_view: ONLINE_RETURNS.ANALYTICS.QUICK_RETURNS_SV
    execution_environment:
      type: warehouse
      warehouse: COMPUTE_WH
$$;

### Ask the quick agent

Run the **`quick_ask`** cell. It sends the agent this question and prints the answer:

> *"Which product categories have the highest and lowest return rate?"*

**The agent's answer:** Apparel has the highest return rate, at about **45%**, and Footwear the lowest, at about **14%**. The agent may even merge the messy category spellings for you.

The answer looks reasonable, and it's wrong: the semantic view trusts the export's `order_status`, which STEP 3 showed is only one of two records of returns. You'll come back to this answer in STEP 12.

In [ ]:
SELECT c.value:text::STRING AS answer
FROM TABLE(FLATTEN(TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'ONLINE_RETURNS.ANALYTICS.RETURNS_AGENT',
  $${"messages": [{"role": "user", "content": [{"type": "text",
     "text": "Which product categories have the highest and lowest return rate?"}]}]}$$,
  FALSE
)):content)) c
WHERE c.value:type::STRING = 'text';

## STEP 5: Find the problem with CoCo

The quick agent trusted one of the two records from STEP 3. To find out what it missed, you'll use **CoCo**.

### What is CoCo?

CoCo (Cortex Code) is Snowflake's AI assistant, built into Snowsight and Workspaces. You tell it what you want in plain language, and it works out the steps: it finds the right tables, writes SQL or Python, runs it, and explains what it found. It can also fix a query that fails. You'll use it to investigate here, and to build tables and a search service in STEPs 6–8.

**How to use it**

- Open the **CoCo** panel from the Workspace toolbar and type a request in the message box.
- Name the tables you care about, or type `@` to add a table, schema, or file to the conversation.
- Before CoCo runs anything, it shows you what it plans to do and asks for approval. You can allow that one call, allow that tool for the rest of the chat, or ask for changes instead.
- When CoCo edits a file, it shows the change side by side before it's applied.

**Why use it here**

Finding where two records disagree means exploring unfamiliar tables, matching up different spellings, and writing joins. CoCo does that legwork in a minute, so you can spend your time deciding whether the answer makes sense.

**Why you can trust it**

- **It works under your role.** CoCo can only see and change what your role is allowed to. Snowflake's access controls and policies apply to everything it runs.
- **You approve what runs.** By default, CoCo asks before each step, and you can read the SQL first.
- **It uses your account's real context.** It reads your table and column details and Snowflake's documentation, so it isn't guessing about your data.
- **You can check its work.** Every result in this lab has an expected answer to compare against.

> CoCo and the agents phrase things differently every time. Don't compare wording; compare the numbers.

### Compare the two records

Open the **CoCo** panel and send:

> *"Compare ONLINE_RETURNS.RAW.ORDERS with the return events in ONLINE_RETURNS.RAW.RETURN_EVENTS. For each product category, show two columns: export_returned (lines whose order_status says returned) and events_refunded_or_exchanged (lines whose latest return event is a refund or exchange). Group the different category spellings together. Don't change any data."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Compare `ONLINE_RETURNS.RAW.ORDERS` with … `RAW.RETURN_EVENTS`" | Names the exact tables, so CoCo doesn't have to guess which data you mean |
| "For each product category, show two columns" | Describes the shape of the result, so you get a table you can check |
| The definitions in parentheses | Says exactly what each number counts. Without them, "returned" could mean several things. |
| "Group the different category spellings together" | Tells CoCo about the messy values you saw in STEP 3 |
| "Don't change any data" | Sets a limit: this step only looks, it doesn't build anything |

<details>
<summary>Expected output (check against this)</summary>

| Category | export_returned | events_refunded_or_exchanged |
|---|---:|---:|
| Accessories | 125 | 75 |
| Apparel | 250 | 150 |
| Footwear | **75** | **200** |
| Home | 125 | 75 |

</details>

Compare the two columns category by category. For Footwear, the export says **75** returns but the events say **200**. That's why the quick agent called Footwear the best category. In every other category the export shows *more* returns than actually happened.

### Look at two order lines up close

Run the **`peek_raw`** cell. It shows two order lines next to the full history of their return events.

In [ ]:
WITH events AS (
  SELECT
    value:order_line_id::VARCHAR AS order_line_id,
    value:event_timestamp::TIMESTAMP_TZ AS event_timestamp,
    value:return_status::VARCHAR AS return_status,
    value:source::VARCHAR AS source
  FROM ONLINE_RETURNS.RAW.RETURN_EVENTS,
       LATERAL FLATTEN(input => payload)
)
SELECT o.order_line_id, o.category, o.region, o.order_status AS export_says,
       e.event_timestamp, e.return_status, e.source
FROM ONLINE_RETURNS.RAW.ORDERS o
JOIN events e USING (order_line_id)
WHERE o.order_line_id IN ('L-2006', 'L-2020')
ORDER BY o.order_line_id, e.event_timestamp;

Things to notice:

- **L-2006 (Apparel):** the export says `Returned`, but the warehouse rejected the return and the customer kept the item. It is **not** a return. The export marks a line "returned" as soon as someone *asks*. That's why the export overcounts Apparel, Accessories, and Home.
- **L-2020 (Footwear, EMEA):** the export says `delivered`, but the events show the customer requested a return, the warehouse received it, and support issued a refund. It **is** a return. Footwear returns go through a separate warehouse system that never updates the export, so the export undercounts Footwear.
- Some lines were never delivered (`shipped`, `cancelled`). They can't be returned, so they shouldn't count in the rate.

The fix is to agree on what "a return" means, and build that rule into the data before any agent sees it.

## STEP 6: Define a return and clean the order lines

The rule for this lab:

- **A return** is an order line whose **latest** return event is a refund or an exchange. Requested or rejected returns don't count.
- **Return rate** = returned lines ÷ delivered lines.
- Categories and regions get one spelling each. Prices and dates get real types.

Open the **CoCo** panel and send this prompt. Compare CoCo's SQL with the expected output below. If it does the same thing, click **Allow** to run it. It doesn't have to match word for word, but the table and column names must match, because later cells use them.

> *"Create ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS with one row per order line from ONLINE_RETURNS.RAW.ORDERS. Map category to Footwear, Apparel, Accessories, or Home, and region to US East, US West, Canada, or EMEA. Convert item_price to a number and order_date to a date column named order_dt. Using each line's latest event in ONLINE_RETURNS.RAW.RETURN_EVENTS, add latest_return_status (requested, received, refunded, exchanged, or rejected), is_returned (true only for refunded or exchanged), and is_delivered (true when the export says delivered or returned)."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Create `ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS` with one row per order line" | Names the table to build and its grain, so nothing gets duplicated or dropped |
| "Map category to Footwear, Apparel, Accessories, or Home…" | Lists the only allowed values, so every messy spelling lands in one of them |
| "Convert item_price to a number and order_date to a date column named order_dt" | Turns text into real types you can sum and filter on |
| "Using each line's latest event…" | Puts the rule for a return into the table, so no one has to apply it later |
| The exact column names (`order_dt`, `latest_return_status`, `is_returned`, `is_delivered`) | Later cells and the semantic view use these names, so they have to match |

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE TABLE ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS AS
WITH events AS (
  SELECT
    value:order_line_id::VARCHAR        AS order_line_id,
    value:event_timestamp::TIMESTAMP_TZ AS event_timestamp,
    CASE LOWER(value:return_status::VARCHAR)
      WHEN 'return_requested' THEN 'requested'
      WHEN 'return requested' THEN 'requested'
      WHEN 'received'         THEN 'received'
      WHEN 'refunded'         THEN 'refunded'
      WHEN 'refund_issued'    THEN 'refunded'
      WHEN 'exchanged'        THEN 'exchanged'
      WHEN 'exchange_shipped' THEN 'exchanged'
      WHEN 'rejected'         THEN 'rejected'
      WHEN 'return_rejected'  THEN 'rejected'
    END AS return_status
  FROM ONLINE_RETURNS.RAW.RETURN_EVENTS,
       LATERAL FLATTEN(input => payload)
),
latest AS (
  SELECT order_line_id, return_status
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_line_id ORDER BY event_timestamp DESC) = 1
)
SELECT
  o.order_line_id,
  o.order_id,
  o.customer_id,
  CASE
    WHEN LOWER(TRIM(o.category)) IN ('footwear', 'shoes')                        THEN 'Footwear'
    WHEN LOWER(TRIM(o.category)) IN ('apparel', 'clothing')                      THEN 'Apparel'
    WHEN LOWER(TRIM(o.category)) IN ('accessories', 'accessory', 'acc')          THEN 'Accessories'
    WHEN LOWER(TRIM(o.category)) IN ('home', 'home goods', 'home & living')      THEN 'Home'
  END AS category,
  CASE
    WHEN TRIM(o.region) IN ('US East', 'US-East', 'United States East', 'USA East') THEN 'US East'
    WHEN TRIM(o.region) IN ('US West', 'US-West', 'West Coast')                     THEN 'US West'
    WHEN TRIM(o.region) IN ('Canada', 'CA', 'CAN')                                  THEN 'Canada'
    WHEN TRIM(o.region) IN ('EMEA', 'Europe', 'Europe/ME/Africa')                   THEN 'EMEA'
  END AS region,
  TRY_TO_NUMBER(REPLACE(REPLACE(o.item_price, '$', ''), ',', ''), 12, 2) AS item_price,
  COALESCE(
    TRY_TO_DATE(o.order_date, 'MM/DD/YYYY'),
    TRY_TO_DATE(o.order_date, 'YYYY-MM-DD'),
    TRY_TO_DATE(o.order_date, 'MON DD YYYY'),
    TRY_TO_DATE(o.order_date, 'YYYY/MM/DD')
  ) AS order_dt,
  o.order_status AS source_order_status,
  LOWER(o.order_status) IN ('delivered', 'returned') AS is_delivered,
  l.return_status AS latest_return_status,
  COALESCE(l.return_status IN ('refunded', 'exchanged'), FALSE) AS is_returned
FROM ONLINE_RETURNS.RAW.ORDERS o
LEFT JOIN latest l USING (order_line_id);
```

</details>

### Check the numbers

Run the **`check1`** cell. Each number comes from somewhere you can trace:

| Column | Expect | Where it comes from |
|---|---:|---|
| `order_lines` | **2,000** | Every line in the export, one row each |
| `unmapped_rows` | **0** | Every category, region, price, and date was converted; anything left as NULL would show up here |
| `delivered_lines` | **1,800** | 2,000 lines minus the 100 `shipped` and 100 `cancelled` lines |
| `returned_lines` | **500** | The *events* total from STEP 5: 75 + 150 + 200 + 75. The export's total was 575. |
| `return_rate_pct` | **27.8** | 500 ÷ 1,800 |
| `returned_value` | **46958.00** | The sum of `item_price` on those 500 lines |

If a number is off, ask CoCo: *"Why doesn't this table have 500 returned lines?"*

In [ ]:
SELECT
  COUNT(*)                                                        AS order_lines,
  COUNT_IF(category IS NULL OR region IS NULL OR item_price IS NULL OR order_dt IS NULL) AS unmapped_rows,
  COUNT_IF(is_delivered)                                          AS delivered_lines,
  COUNT_IF(is_returned)                                           AS returned_lines,
  ROUND(100 * DIV0(COUNT_IF(is_returned), COUNT_IF(is_delivered)), 1) AS return_rate_pct,
  SUM(IFF(is_returned, item_price, 0))                            AS returned_value
FROM ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS;

## STEP 7: Translate and tag comments with AI

The numbers now say *where* returns happen. The comments say *why*, but a third of them aren't in English, and free text can't be counted. Two AI functions fix that. They run inside a normal SQL query, one call per row.

**`AI_TRANSLATE(text, source_language, target_language)`** returns the text in the target language. Passing `''` as the source language tells it to detect the language itself, so `AI_TRANSLATE(comment, '', 'en')` turns German, French, or Spanish into English and leaves English as it is.

**`AI_CLASSIFY(text, categories, config)`** picks the category that fits the text best:

- `categories` is your list of allowed answers. Each can have a short `description` that tells the model what belongs there, such as *"The item is too big, too small, too tight, or does not fit"* for `Wrong size`.
- `config` can hold a `task_description`, one sentence on what you're classifying.
- It returns an object like `{"labels": ["Wrong size"]}`, so the SQL takes `:labels[0]` to get the label as text.

Because the model can only answer with one of your labels, every comment ends up in a bucket you can count. Snowflake recommends English input for `AI_CLASSIFY`, so the lab translates first and classifies `comment_english`.

AI functions are billed by the tokens they read and write. The comment, and the category list with its descriptions, all count, so 775 short comments cost very little.

### Try it on one comment

Run the **`ai_try`** cell. It takes one German comment from the lab data, translates it, and then classifies the English text into one of three labels.

In [ ]:
SELECT
  'Die Schuhe fallen eine ganze Nummer zu klein aus.' AS comment,
  AI_TRANSLATE(comment, '', 'en') AS comment_english,
  AI_CLASSIFY(
    comment_english,
    [
      {'label': 'Wrong size', 'description': 'The item is too big, too small, too tight, or does not fit'},
      {'label': 'Damaged or defective', 'description': 'The item arrived broken, scratched, or failed after little use'},
      {'label': 'Changed mind', 'description': 'Nothing is wrong with the item; the customer no longer wants it'}
    ],
    {'task_description': 'Classify the main reason a customer gives for returning an online order.'}
  ):labels[0]::STRING AS return_reason;

Expected result:

| comment | comment_english | return_reason |
|---|---|---|
| Die Schuhe fallen eine ganze Nummer zu klein aus. | The shoes run a whole size too small. | Wrong size |

`AI_TRANSLATE` detected German because the source language is `''`. `AI_CLASSIFY` could only answer with one of the three labels, and it picked the one whose description matched. The English wording may differ slightly; the label is what to check.

Next, CoCo applies the same two functions to all 775 comments, with the full list of six labels.

### Run the AI functions

Send this prompt to **CoCo**:

> *"Create ONLINE_RETURNS.CURATED.RETURN_COMMENT_INSIGHTS from the comments in ONLINE_RETURNS.RAW.RETURN_COMMENTS. Keep order_line_id and the original comment, translate it to English with AI_TRANSLATE into comment_english, then use AI_CLASSIFY on comment_english to set return_reason to Wrong size, Damaged or defective, Not as described, Arrived late, Changed mind, or Other. Then create ONLINE_RETURNS.CURATED.RETURN_REVIEW: every row of ORDER_LINE_RETURNS plus comment, comment_english, and return_reason."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Keep order_line_id and the original comment" | Lets you join back to the order lines and quote customers in their own words |
| "translate it to English with AI_TRANSLATE into comment_english" | Names the function and the output column, so CoCo uses the one you just tried |
| "use AI_CLASSIFY on comment_english" | Sets the order: translate first, then classify the English |
| The list of six reasons | The only labels allowed, so every comment can be counted |
| "every row of ORDER_LINE_RETURNS plus…" | Keeps all 2,000 lines, including ones without a comment, so the numbers from `check1` still hold |

Check that CoCo's SQL translates first and classifies the translation. The query runs AI on 775 comments, so it takes longer than a normal query (about 15–30 seconds).

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE TABLE ONLINE_RETURNS.CURATED.RETURN_COMMENT_INSIGHTS AS
WITH comments AS (
  SELECT
    value:order_line_id::VARCHAR          AS order_line_id,
    value:comment_timestamp::TIMESTAMP_TZ AS comment_timestamp,
    value:comment::VARCHAR                AS comment
  FROM ONLINE_RETURNS.RAW.RETURN_COMMENTS,
       LATERAL FLATTEN(input => payload)
),
translated AS (
  SELECT order_line_id, comment_timestamp, comment,
         AI_TRANSLATE(comment, '', 'en') AS comment_english
  FROM comments
)
SELECT
  order_line_id,
  comment_timestamp,
  comment,
  comment_english,
  AI_CLASSIFY(
    comment_english,
    [
      {'label': 'Wrong size',           'description': 'The item is too big, too small, too tight, or does not fit'},
      {'label': 'Damaged or defective', 'description': 'The item arrived broken, scratched, or failed after little use'},
      {'label': 'Not as described',     'description': 'The color, material, or look does not match the listing or photos'},
      {'label': 'Arrived late',         'description': 'The item arrived too late to be useful'},
      {'label': 'Changed mind',         'description': 'Nothing is wrong with the item; the customer no longer wants it'},
      {'label': 'Other',                'description': 'No return reason given, such as a question about refund status'}
    ],
    {'task_description': 'Classify the main reason a customer gives for returning an online order.'}
  ):labels[0]::STRING AS return_reason
FROM translated;

CREATE OR REPLACE TABLE ONLINE_RETURNS.CURATED.RETURN_REVIEW AS
SELECT
  r.*,
  c.comment,
  c.comment_english,
  c.return_reason
FROM ONLINE_RETURNS.CURATED.ORDER_LINE_RETURNS r
LEFT JOIN ONLINE_RETURNS.CURATED.RETURN_COMMENT_INSIGHTS c USING (order_line_id);
```

</details>

### Look at what the AI decided

The biggest return problem is Footwear in EMEA, so look there. Run the **`review_labels`** cell. It shows each distinct comment on a returned Footwear line in EMEA, how many lines used it, its translation, and its label.

You should see **8** distinct comments, all in German, French, or Spanish. Six are about size (*"Die Schuhe fallen eine ganze Nummer zu klein aus."*) and are tagged `Wrong size`. Two are about soles coming off and are tagged `Damaged or defective`. Read the translations and decide whether you agree with each label.

In [ ]:
SELECT comment, comment_english, return_reason, COUNT(*) AS lines
FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW
WHERE is_returned AND category = 'Footwear' AND region = 'EMEA'
GROUP BY comment, comment_english, return_reason
ORDER BY lines DESC;

### Before and after AI

Here's what the AI bought you. Run the **`before_keywords`** cell. It's what you might try without AI: search the 100 returned Footwear/EMEA comments for English size words.

In [ ]:
-- Before AI: count comments that mention size in English.
SELECT COUNT(*) AS returned_lines_with_comment,
       COUNT_IF(comment ILIKE ANY ('%size%', '%small%', '%big%', '%tight%', '%fit%')) AS mention_size_in_english
FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW
WHERE is_returned AND category = 'Footwear' AND region = 'EMEA';

Expect **100** comments and **0** matches. Every one is written in another language, so a keyword search finds nothing.

Now run the **`after_ai`** cell, which counts the AI labels for the same 100 lines.

In [ ]:
-- After AI: count the same lines by AI-tagged reason.
SELECT return_reason, COUNT(*) AS returned_lines
FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW
WHERE is_returned AND category = 'Footwear' AND region = 'EMEA'
GROUP BY return_reason
ORDER BY returned_lines DESC;

Expect **Wrong size: 75** and **Damaged or defective: 25**. Same comments, but now you can count them: three out of four Footwear returns in EMEA are about size.

### Check the answer before handing it to an agent

If these numbers are wrong, an agent will only give you the wrong answer faster. Run the **`check2`** cell.

| Column | Expect | Where it comes from |
|---|---:|---|
| `order_lines`, `delivered_lines`, `returned_lines`, `returned_value` | **2,000 / 1,800 / 500 / 46958.00** | Same as `check1`. Joining the comments must not add or drop lines. |
| `wrong_size_returns` | **175** | Returned lines tagged `Wrong size` across all categories and regions, including the 75 you just saw in Footwear/EMEA |
| `invalid_labels` | **0** | Every label is one of your six categories |

AI labels can vary a little, so `wrong_size_returns` may be off by one or two. A bigger gap means CoCo's labels or descriptions differ from the expected output.

In [ ]:
SELECT
  COUNT(*)                                             AS order_lines,
  COUNT_IF(is_delivered)                               AS delivered_lines,
  COUNT_IF(is_returned)                                AS returned_lines,
  SUM(IFF(is_returned, item_price, 0))                 AS returned_value,
  COUNT_IF(is_returned AND return_reason = 'Wrong size') AS wrong_size_returns,
  COUNT_IF(return_reason IS NOT NULL AND return_reason NOT IN
    ('Wrong size', 'Damaged or defective', 'Not as described', 'Arrived late', 'Changed mind', 'Other')) AS invalid_labels
FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW;

## STEP 8: Make the comments searchable (Cortex Search)

An agent needs two kinds of help. For numbers, it writes SQL. For "show me what customers said", it needs to find the right comments, and a customer who says *"Me quedan pequeñas"* never typed the word "small".

**Cortex Search** is a search service you build over a text column. It combines meaning-based (vector) search with keyword search, then reranks the results, so a search for "shoes too small" finds comments that say the same thing in other words. You choose:

- the column to search: `comment_english`
- the columns you can filter on (**attributes**): category, region, return reason
- extra columns to return with each result: the original `comment` and `order_line_id`

Snowflake keeps the service in sync with the table, refreshing on the schedule you set with `TARGET_LAG`.

Send this prompt to **CoCo**:

> *"Create Cortex Search service ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH on the comment_english column of ONLINE_RETURNS.CURATED.RETURN_REVIEW, only for rows that have a comment. Use category, region, return_reason, and latest_return_status as attributes, and also return order_line_id and comment. Use warehouse COMPUTE_WH and a target lag of 1 day."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "on the comment_english column" | Picks the text to search. Searching the English translation lets one query match comments in every language. |
| "only for rows that have a comment" | Leaves out order lines with nothing to search |
| "Use category, region, return_reason, and latest_return_status as attributes" | The columns the agent can filter on, such as region = EMEA |
| "also return order_line_id and comment" | Each result comes back with its ID and the original-language text to quote |
| "warehouse COMPUTE_WH and a target lag of 1 day" | What runs the refreshes, and how fresh the service stays |

The service takes a minute or so to build.

> **Short on time?** Paste the expected output into the empty cell below and run it.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE CORTEX SEARCH SERVICE ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH
  ON comment_english
  ATTRIBUTES category, region, return_reason, latest_return_status
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
  AS (
    SELECT order_line_id, comment, comment_english, category, region, return_reason, latest_return_status
    FROM ONLINE_RETURNS.CURATED.RETURN_REVIEW
    WHERE comment IS NOT NULL
  );
```

</details>

### Check the search service

Run the **`check_search`** cell. It searches for *"shoes too small"* and returns the top five matches. Expect size complaints about shoes, several of them from EMEA, with the original comment in German, French, or Spanish next to its English translation. None of them contain the word "small" in their original language.

In [ ]:
SELECT r.value:order_line_id::STRING AS order_line_id,
       r.value:region::STRING AS region,
       r.value:comment::STRING AS comment,
       r.value:comment_english::STRING AS comment_english
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH',
  '{"query": "shoes too small", "columns": ["order_line_id", "comment", "comment_english", "region"], "limit": 5}'
)):results)) r;

## Your data is ready

You've finished the notebook work: clean order lines, AI-tagged comments, and a search service. The rest of the lab happens in Snowsight, where you'll look at what you built and give it to an agent. You won't need to come back to this notebook until teardown.

The steps continue below and in the guide; follow along from either.

## STEP 9: Look at the search service in Snowsight

1. In Snowsight, navigate to **AI & ML » Cortex Search**.
2. Select **RETURN_COMMENTS_SEARCH** (database `ONLINE_RETURNS`, schema `ANALYTICS`).
3. Look at the details page. You should see:
   - the **search column**, `COMMENT_ENGLISH`
   - the **attributes** you can filter on: `CATEGORY`, `REGION`, `RETURN_REASON`, `LATEST_RETURN_STATUS`
   - the warehouse and target lag
4. Open the playground on the service page and search for *"the sole came off"*. You should get the Damaged or defective comments, including *"Die Sohle hat sich nach dem ersten Tragen gelöst."* and *"La suela se despegó después de una semana."*
5. Add a filter `REGION = EMEA` and search for *"too small"*. Every result should be a Footwear comment in German, French, or Spanish.

This is the tool the agent will call when someone asks *why*.

## STEP 10: Write down your definitions (semantic view)

**Cortex Analyst** is Snowflake's text-to-SQL engine: it turns a question into SQL and runs it. To write correct SQL, it needs to know what your columns mean, and that's what a **semantic view** holds:

- **Dimensions** are what you group and filter by: category, region, return reason.
- **Metrics** are the calculations, defined once: *returned lines*, *delivered lines*, *returned value*, *return rate*.
- **Verified queries** are question-and-SQL pairs you've confirmed are right. Analyst uses them as worked examples.

The quick agent's semantic view defined a return the wrong way. This one uses the rule from STEP 6.

1. In Snowsight, navigate to **AI & ML » Analyst**.
2. Click **Create in Workspaces** in the top right.
3. Click **Create with CoCo**. Snowsight opens a new `.sv.yaml` file, and the CoCo panel asks what to build.
4. Send CoCo this prompt:

> *"Create semantic view RETURNS_SV in ONLINE_RETURNS.ANALYTICS over ONLINE_RETURNS.CURATED.RETURN_REVIEW, with order_line_id as the primary key. Dimensions: category, region, return_reason, latest_return_status, is_delivered, is_returned. Metrics: delivered_lines (count where is_delivered), returned_lines (count where is_returned), returned_value (sum of item_price where is_returned), and return_rate (returned_lines divided by delivered_lines). Add verified queries for: return rate by category and region; overall return rate and returned value; and why customers return items, by category, region, and return_reason."*

   **What this prompt asks for**

   | Part of the prompt | Why it's there |
   |---|---|
   | "over `RETURN_REVIEW`, with order_line_id as the primary key" | Names the one table to describe and what makes each row unique |
   | "Dimensions: …" | What people can group and filter by |
   | "Metrics: … (count where is_delivered) …" | Each calculation, spelled out once, so Analyst never has to guess what "return rate" means |
   | "Add verified queries for: …" | The questions you expect most, so Analyst has checked examples to follow |

5. Allow CoCo to create the draft. This creates an editable draft; it doesn't publish the view yet.
6. Review the draft. Confirm it contains:
   - **Dimensions:** `CATEGORY`, `REGION`, `RETURN_REASON`, `LATEST_RETURN_STATUS`, `IS_DELIVERED`, `IS_RETURNED`
   - **Metrics:** `DELIVERED_LINES`, `RETURNED_LINES`, `RETURNED_VALUE`, `RETURN_RATE`. Open `RETURN_RATE` and check that it divides returned lines by **delivered** lines, not all lines.
   - **Verified queries:** three, one for each question in the prompt

   CoCo may add descriptions or synonyms. That's fine. If a metric is missing or defined differently, ask CoCo to fix it before publishing (*"return_rate should divide returned_lines by delivered_lines"*).
7. Click **Publish** in the top right. Confirm **Name** `RETURNS_SV`, **Database** `ONLINE_RETURNS`, and **Schema** `ANALYTICS`, then click **Publish**.
8. Test it. In the Analyst playground for `RETURNS_SV`, ask *"What is our overall return rate and how much revenue was returned?"* Expect **500** returned of **1,800** delivered lines, **27.8%**, and **$46,958.00**, the same numbers as `check1`. Click the SQL to see the query Analyst wrote.

<details>
<summary>Expected definition (for comparison, or to create the view with SQL if CoCo's differs)</summary>

```sql
CREATE OR REPLACE SEMANTIC VIEW ONLINE_RETURNS.ANALYTICS.RETURNS_SV
  TABLES (
    returns AS ONLINE_RETURNS.CURATED.RETURN_REVIEW
      PRIMARY KEY (order_line_id)
      COMMENT = 'One row per Q1 2026 order line. A return is a line whose latest return status is refunded or exchanged.'
  )
  DIMENSIONS (
    returns.category AS category COMMENT = 'Footwear, Apparel, Accessories, or Home',
    returns.region AS region COMMENT = 'US East, US West, Canada, or EMEA',
    returns.latest_return_status AS latest_return_status COMMENT = 'requested, received, refunded, exchanged, or rejected',
    returns.return_reason AS return_reason COMMENT = 'AI-derived reason from the customer comment. Other means no reason given.',
    returns.is_delivered AS is_delivered COMMENT = 'TRUE if the line was delivered (shipped and cancelled lines are FALSE)',
    returns.is_returned AS is_returned COMMENT = 'TRUE only if the latest return status is refunded or exchanged'
  )
  METRICS (
    returns.delivered_lines AS COUNT_IF(is_delivered) COMMENT = 'Delivered order lines, the denominator of the return rate',
    returns.returned_lines AS COUNT_IF(is_returned) COMMENT = 'Returned order lines (refunded or exchanged only)',
    returns.returned_value AS SUM(IFF(is_returned, item_price, 0)) COMMENT = 'USD value of returned order lines',
    return_rate AS DIV0(returns.returned_lines, returns.delivered_lines) COMMENT = 'Returned lines divided by delivered lines'
  )
  COMMENT = 'Northwind Outfitters returns, Q1 2026'
  AI_VERIFIED_QUERIES (
    rate_by_category_region AS (
      QUESTION 'What is our return rate by product category and region?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT category, region, COUNT_IF(is_delivered) AS delivered_lines, COUNT_IF(is_returned) AS returned_lines, ROUND(DIV0(COUNT_IF(is_returned), COUNT_IF(is_delivered)), 3) AS return_rate FROM returns GROUP BY category, region ORDER BY return_rate DESC'
    ),
    overall_rate AS (
      QUESTION 'What is our overall return rate and how much revenue was returned?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT COUNT_IF(is_delivered) AS delivered_lines, COUNT_IF(is_returned) AS returned_lines, ROUND(DIV0(COUNT_IF(is_returned), COUNT_IF(is_delivered)), 3) AS return_rate, SUM(IFF(is_returned, item_price, 0)) AS returned_value FROM returns'
    ),
    reasons AS (
      QUESTION 'Why are customers returning items?'
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT category, region, return_reason, COUNT(*) AS returned_lines, SUM(item_price) AS returned_value FROM returns WHERE is_returned GROUP BY category, region, return_reason ORDER BY returned_lines DESC, returned_value DESC'
    )
  );
```

</details>

## STEP 11: Fix the agent

An **agent** reads a question, decides which tool can answer it, calls the tool, and writes the answer. The `RETURNS_AGENT` you created in STEP 4 has one tool, pointed at the quick semantic view. You'll edit it so it has two tools:

- **Cortex Analyst** over `RETURNS_SV` for counts, rates, and values
- **Cortex Search** over `RETURN_COMMENTS_SEARCH` for what customers actually said

Its **instructions** tell it when to use each one and how to answer.

1. In Snowsight, navigate to **AI & ML » Agents** (labeled **Agent Studio** on some accounts).
2. Select **RETURNS_AGENT** (database `ONLINE_RETURNS`, schema `ANALYTICS`).
3. Click **Edit**.
4. On the general settings, set:
   - **Description:** `Answers questions about Northwind Outfitters returns: return rates and returned value by category and region, and what customers said about why they returned items.`
   - **Example questions:** `Which product categories have the highest return rate?` and `Why are customers returning footwear in EMEA?`
5. Under **Tools**, remove the `quick_returns` tool. Then add **Cortex Analyst** (**+ Add**, or **Add semantic view**):
   - **Semantic view:** `ONLINE_RETURNS.ANALYTICS.RETURNS_SV`
   - **Name:** `returns_analytics`
   - **Warehouse:** `COMPUTE_WH`
   - **Description:** `Return counts, rates, values, and AI-tagged reasons by category and region for Q1 2026 order lines.`
6. Under **Tools**, add a **Cortex Search** service (**+ Add** next to **Cortex Search Services**):
   - **Search service:** `ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH`
   - **Name:** `customer_comments`
   - **Description:** `Customer return comments in their original language with an English translation, filterable by category, region, and reason.`
   - **ID column** and **Title column:** `ORDER_LINE_ID`
7. Under the orchestration or instructions settings, paste:
   - **Orchestration (planning) instructions:** `Use returns_analytics for any count, rate, value, or ranking question. Use customer_comments to find what customers said, and quote them when you explain a reason.`
   - **Response instructions:** `A return is an order line whose latest return status is refunded or exchanged. Return rate is returned lines divided by delivered lines. State counts, percentages, and USD values precisely. When you quote a customer, quote the COMMENT column exactly as written; if it is not in English, add the English translation from COMMENT_ENGLISH in parentheses. Start with the answer and answer only the question asked.`
8. Click **Save**.
9. Test it in the agent playground on the agent's page. Ask *"Which product categories have the highest return rate?"* Expect **Footwear at 42.1%** first. Expand the response details to see that it called `returns_analytics`.

<details>
<summary>Expected definition (for comparison, or to make the same changes with SQL)</summary>

```sql
CREATE OR REPLACE AGENT ONLINE_RETURNS.ANALYTICS.RETURNS_AGENT
  COMMENT = 'Northwind Outfitters returns agent'
  FROM SPECIFICATION
$$
models:
  orchestration: auto
instructions:
  orchestration: >
    Use returns_analytics for any count, rate, value, or ranking question.
    Use customer_comments to find what customers actually said, and quote them when you explain a reason.
  response: >
    A return is an order line whose latest return status is refunded or exchanged; requested and rejected
    returns are not returns. Return rate is returned lines divided by delivered lines. State counts,
    percentages, and USD values precisely. When you quote a customer, quote the COMMENT column exactly as
    written; if it is not in English, follow it with the English translation from COMMENT_ENGLISH in parentheses.
    Start the answer with the answer itself, not with notes about what you plan to do. Answer only the question asked.
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: returns_analytics
      description: Return counts, rates, values, and AI-derived reasons by category and region for Q1 2026 order lines.
  - tool_spec:
      type: cortex_search
      name: customer_comments
      description: Customer return comments in the original language with an English translation, filterable by category, region, and reason.
tool_resources:
  returns_analytics:
    semantic_view: ONLINE_RETURNS.ANALYTICS.RETURNS_SV
    execution_environment:
      type: warehouse
      warehouse: COMPUTE_WH
  customer_comments:
    search_service: ONLINE_RETURNS.ANALYTICS.RETURN_COMMENTS_SEARCH
    max_results: 5
    id_column: ORDER_LINE_ID
    title_column: ORDER_LINE_ID
    columns_and_descriptions:
      COMMENT:
        description: "The customer's comment exactly as written, in its original language (English, German, French, or Spanish)."
        type: string
        searchable: false
        filterable: false
      COMMENT_ENGLISH:
        description: The comment translated into English. This is the searchable text.
        type: string
        searchable: true
        filterable: false
      REGION:
        description: "Customer region: US East, US West, Canada, or EMEA."
        type: string
        searchable: false
        filterable: true
      CATEGORY:
        description: "Product category: Footwear, Apparel, Accessories, or Home."
        type: string
        searchable: false
        filterable: true
$$;
```

</details>

## STEP 12: Ask in CoWork

**Snowflake CoWork** is where anyone on the team chats with your agents, without opening a notebook or writing SQL. Agents you can use show up there automatically.

1. In Snowsight, navigate to **AI & ML » Snowflake CoWork**.
2. Select **RETURNS_AGENT** from the agent list.
3. Ask the same question you asked in STEP 4, then two follow-ups in the same conversation:
   1. *"Which product categories have the highest and lowest return rate?"*
   2. *"Which category and region should we look at first?"*
   3. *"Why are customers returning footwear in EMEA? Show me what they said."*

**Expected:**

| Question | Expected answer | Where it comes from |
|---|---|---|
| 1 | Footwear highest at **42.1%** (200 of 475 delivered); Home lowest at **17.6%** | `returned_lines` ÷ `delivered_lines` per category from `RETURNS_SV` |
| 2 | **Footwear in EMEA**: **100** of **150** delivered lines returned, **66.7%** | The same metrics by category and region |
| 3 | About **3 in 4** are **Wrong size** (75 of 100), quoted in the original language with a translation, such as *"Die Schuhe fallen eine ganze Nummer zu klein aus."* (*"The shoes are a whole size too small."*) | The AI labels from STEP 7, and quotes found by Cortex Search |

Click the steps in each answer to see which tool the agent used: Cortex Analyst for questions 1 and 2, and both tools for question 3.

In STEP 4, the same agent answered question 1 with Apparel worst (about 45%) and Footwear best (about 14%). The only difference is the work you did on the data.

Try rewording question 1, for example *"What gets sent back most?"*, and check that RETURNS_AGENT's numbers don't change.

## What you built

- Raw tables that keep exactly what the source systems sent
- A curated table with one clear definition of a return
- AI-translated, AI-tagged customer comments you can count and search
- A Cortex Search service that finds comments by meaning, in any language
- A semantic view with metrics and verified queries, so every question uses the same definitions
- An agent that answers with numbers *and* customer quotes, available to anyone in CoWork

Your agent used the same AI in STEP 4 and STEP 12. What made the difference was the work you did on the data in between.

## STEP 13: Clean up

Run this only after you've finished the lab and any validation checks. Run the **`teardown`** cell, or run the same SQL in a SQL worksheet. Dropping the database removes everything inside it: tables, semantic views, the search service, and the agent. The second statement removes the Git API integration you created in setup. After that, the workspace can no longer pull from GitHub, so run teardown last.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Drops the tables, semantic views, search service, and agent
DROP DATABASE IF EXISTS ONLINE_RETURNS;

-- Removes the Git API integration created for the workspace
DROP API INTEGRATION IF EXISTS GITHUB_RETURNS_LAB;